# SC5006 Project 1 — CountGD Component Analysis and Open-World Counting

**Team size:** 1–5 students  
**Foundation model:** [CountGD](https://github.com/niki-amini-naieni/CountGD), NeurIPS 2024  
**Test images:** `women.jpg` and `car.jpg`  
**Required evidence:** component shapes/visualizations, live-model predictions, quantitative results, and failure analysis

The project studies five linked CountGD components. A five-person team should assign one primary owner to each component; smaller teams combine roles. Every member must understand the integrated pipeline and speak during the oral presentation.

| Role | Assessed component | Main evidence |
|---|---|---|
| A | Image encoder | multi-scale features and combined 1/8-scale feature map |
| B | Text encoder | tokenization, masks, and projected BERT tokens |
| C | ROIAlign | pixel-space exemplar box and extracted visual token |
| D | Similarity matrix | query–prompt logits and padding mask |
| E | Top-K selection | ranked encoder proposals and prompt IDs |

The shared integration experiment runs the official CountGD checkpoint on women (text prompt) and cars (text-only, exemplar-only, and multimodal prompts), then applies the official confidence threshold and counting rule.

## Deliverables

Submit three files named with your team name:

1. `TeamXX_SC5006_Project1.ipynb` — completed code, outputs, plots, and brief interpretations.
2. `TeamXX_SC5006_Project1_Report.pdf` — Title, Authors, Introduction, Related Work, Method, Experiments, Results and Discussion, Conclusion, References, and contribution statement.
3. `TeamXX_SC5006_Project1_Presentation.pptx` — a 10-minute oral presentation; 1-2 members do the presentation; but all members must join the Q&A.

The report should be 6–8 pages excluding references/appendix. The presentation should contain 8–15 content slides. Results must distinguish live-model evidence from cached reference outputs.


## 0. Environment and model setup

Run the cell below from the project folder. It downloads the official CountGD FSC-147 checkpoint and BERT encoder only when missing, verifies the checkpoint SHA-256, loads all model weights, and prints the active paths and device. Internet access is needed only for the first run.


In [ ]:
from pathlib import Path
import hashlib, importlib, os, subprocess, sys, time, warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
import torch
import torch.nn.functional as F

warnings.filterwarnings("ignore")

ROOT = Path.cwd().resolve()
if not (ROOT / "assets/reference/car.jpg").is_file():
    raise FileNotFoundError("Open this notebook from the Project 1 folder.")

ASSETS = ROOT / "assets/reference"
OUTPUT = ROOT / "output/figures"
CHECKPOINTS = ROOT / "checkpoints"
CHECKPOINT = CHECKPOINTS / "checkpoint_fsc147_best.pth"
BERT = CHECKPOINTS / "bert-base-uncased"
COUNTGD_ROOT = ROOT / "vendor/countgd"
OUTPUT.mkdir(parents=True, exist_ok=True)
CHECKPOINTS.mkdir(parents=True, exist_ok=True)

CHECKPOINT_FILE_ID = "1RbRcNLsOfeEbx6u39pBehqsgQiexHHrI"
EXPECTED_CHECKPOINT_SHA256 = "c1bab864b17db345b4c6e3aaabb5765bc2c0a90d0bc8defb5e664a74a50aa126"

def require_package(module, package=None):
    try:
        return importlib.import_module(module)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package or module])
        return importlib.import_module(module)

def download_missing_assets():
    if not CHECKPOINT.is_file():
        print("Downloading the official CountGD FSC-147 checkpoint (1.2 GB)...")
        gdown = require_package("gdown")
        result = gdown.download(id=CHECKPOINT_FILE_ID, output=str(CHECKPOINT), quiet=False)
        if not result or not CHECKPOINT.is_file():
            raise RuntimeError("Official CountGD checkpoint download failed.")
    if not (BERT / "config.json").is_file():
        print("Downloading the official BERT base encoder...")
        huggingface_hub = require_package("huggingface_hub")
        huggingface_hub.snapshot_download(
            repo_id="google-bert/bert-base-uncased", local_dir=str(BERT)
        )

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

download_missing_assets()
CHECKPOINT_SHA256 = sha256_file(CHECKPOINT)
if CHECKPOINT_SHA256 != EXPECTED_CHECKPOINT_SHA256:
    raise RuntimeError(f"Incorrect CountGD checkpoint SHA-256: {CHECKPOINT_SHA256}")
if not (COUNTGD_ROOT / "models/GroundingDINO/groundingdino.py").is_file():
    raise FileNotFoundError("The distributed official source snapshot vendor/countgd is incomplete.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sys.path.insert(0, str(COUNTGD_ROOT))
import datasets_inference.transforms as T
from util.slconfig import SLConfig
from util.misc import nested_tensor_from_tensor_list
from models.GroundingDINO.groundingdino import build_groundingdino
from models.GroundingDINO.bertwarper import generate_masks_with_special_tokens_and_transfer_map
from torchvision.ops import roi_align

TRANSFORM = T.Compose([
    T.RandomResize([800], max_size=1333),
    T.Compose([T.ToTensor(), T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
])

def prepare_case(name, image_name, text="", boxes=()):
    image = Image.open(ASSETS / image_name).convert("RGB")
    clean = " ".join(text.lower().strip().rstrip(".").split())
    boxes = torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)
    if len(boxes) > 3:
        raise ValueError("CountGD supports at most three exemplars")
    if boxes.numel():
        x1, y1, x2, y2 = boxes.unbind(1)
        valid = (x2 > x1) & (y2 > y1) & (x1 >= 0) & (y1 >= 0) & (x2 <= image.width) & (y2 <= image.height)
        if not bool(valid.all()):
            raise ValueError("Exemplars must be positive-area pixel xyxy boxes inside the image")
    mode = "text + exemplar" if clean and len(boxes) else ("text only" if clean else "exemplar only")
    return {"name": name, "image_name": image_name, "image": image,
            "caption": f"{clean} ." if clean else "object .", "boxes": boxes, "mode": mode}

def preprocess_case(case):
    image_tensor, target = TRANSFORM(case["image"], {"exemplars": case["boxes"].clone()})
    return image_tensor, target["exemplars"].to(torch.float32).reshape(-1, 4)

def load_model():
    cfg = SLConfig.fromfile(str(COUNTGD_ROOT / "config/cfg_fsc147_vit_b.py"))
    cfg.device, cfg.text_encoder_type = str(DEVICE), str(BERT)
    model, _, _ = build_groundingdino(cfg)
    message = model.load_state_dict(torch.load(CHECKPOINT, map_location="cpu")["model"], strict=False)
    print("missing keys:", len(message.missing_keys), "| unexpected keys:", len(message.unexpected_keys))
    if message.missing_keys or message.unexpected_keys:
        raise RuntimeError(f"Checkpoint mismatch: {message}")
    return model.to(DEVICE).eval()

CASES = [
    prepare_case("women_text", "women.jpg", text="women"),
    prepare_case("car_text", "car.jpg", text="car"),
    prepare_case("car_exemplar", "car.jpg", boxes=[[55, 72, 151, 218]]),
    prepare_case("car_multimodal", "car.jpg", text="car", boxes=[[55, 72, 151, 218]]),
]
MODEL = load_model()
print("team project root:", ROOT)
print("CountGD source:", COUNTGD_ROOT)
print("checkpoint:", CHECKPOINT)
print("checkpoint SHA-256:", CHECKPOINT_SHA256)
print("BERT:", BERT)
print("device:", DEVICE, "| live official model: True")
for case in CASES:
    print(case["name"], case["mode"], repr(case["caption"]), tuple(case["boxes"].shape))


In [ ]:
for name in ["women.jpg", "car.jpg"]:
    img, _ = preprocess_case(prepare_case(name, name, text="x"))
    print(name, "resized:", tuple(img.shape))
assert tuple(preprocess_case(prepare_case("w", "women.jpg", text="x"))[0].shape) == (3, 800, 1041)
assert tuple(preprocess_case(prepare_case("c", "car.jpg", text="x"))[0].shape) == (3, 800, 1066)

## Task 1 — Image encoder (15%)

Implement `encode_image`. Convert a preprocessed image batch into CountGD's nested-tensor representation, run the official Swin backbone, and combine the multi-scale features into the 1/8-scale feature map used for exemplar extraction.

**TODO 1**

- accept `(B,3,H,W)` or a list of `(3,H,W)` tensors;
- call `nested_tensor_from_tensor_list` and `model.backbone`;
- call `model.combine_features`;
- return the backbone features, positional encodings, and combined feature map; and
- report shapes and visualize mean feature activation.

In [ ]:
def encode_image(model, image_batch):
    if isinstance(image_batch, torch.Tensor):
        image_batch = list(image_batch)
    samples = nested_tensor_from_tensor_list(image_batch).to(DEVICE)
    with torch.inference_mode():
        features, positions = model.backbone(samples)
        combined = model.combine_features(features)
    return features, positions, combined

car_tensor, car_boxes_resized = preprocess_case(CASES[-1])
car_batch = car_tensor.unsqueeze(0)
IMAGE_FEATURES, IMAGE_POSITIONS, COMBINED_FEATURES = encode_image(MODEL, car_batch)

print("preprocessed car image:", tuple(car_tensor.shape))
print("input batch:", tuple(car_batch.shape))

strides = (8, 16, 32)
feature_tensors = []
for level, (feature, position, stride) in enumerate(
    zip(IMAGE_FEATURES, IMAGE_POSITIONS, strides), start=1
):
    tensor, mask = feature.decompose()
    expected_spatial = tuple((size + stride - 1) // stride for size in car_batch.shape[-2:])
    feature_tensors.append(tensor)
    print(
        f"Swin level {level} (1/{stride}): features={tuple(tensor.shape)}, "
        f"mask={tuple(mask.shape)}, positions={tuple(position.shape)}"
    )

print("combined 1/8 features:", tuple(COMBINED_FEATURES.shape))

activation_maps = [
    tensor[0].mean(0).detach().cpu().numpy() for tensor in feature_tensors
]
activation_maps.append(COMBINED_FEATURES[0].mean(0).detach().cpu().numpy())
panel_titles = [
    f"Swin level {level} (1/{stride})\n{tuple(tensor.shape[1:])}"
    for level, (stride, tensor) in enumerate(zip(strides, feature_tensors), start=1)
]
panel_titles.append(f"Combined (1/8)\n{tuple(COMBINED_FEATURES.shape[1:])}")

fig, axes = plt.subplots(1, 4, figsize=(16, 3.8), constrained_layout=True)
for ax, activation, title in zip(axes, activation_maps, panel_titles):
    vmin, vmax = np.percentile(activation, [1, 99])
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        vmin, vmax = float(np.min(activation)), float(np.max(activation))
    if vmax <= vmin:
        vmax = vmin + 1e-6
    image = ax.imshow(activation, cmap="magma", vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.03)
fig.suptitle(
    "Task 1 — CountGD image-encoder channel-wise mean activations for car.jpg",
    fontsize=13,
)
figure_path = OUTPUT / "task1_image_encoder_features.png"
fig.savefig(figure_path, dpi=170, bbox_inches="tight")
plt.show()
print("saved:", figure_path)

## Task 2 — Text encoder (15%)

Implement `encode_text` by reproducing CountGD's caption path: tokenize, build special-token attention structures, run BERT, and project the last hidden state to the shared model dimension.

**TODO 2**

- tokenize a caption batch with padding;
- build self-attention masks and position IDs using the repository helper;
- run `model.bert` and `model.feat_map`;
- return encoded prompt tokens and the valid-token mask; and
- identify which columns correspond to real prompt tokens rather than padding.

In [ ]:
def encode_text(model, captions):
    if isinstance(captions, str):
        captions = [captions]
    if not captions or any(not c.strip() for c in captions):
        raise ValueError("encode_text expects a non-empty list of non-empty captions")
 
    tokenized = model.tokenizer(captions, padding="longest", return_tensors="pt").to(DEVICE)
    self_masks, position_ids, _ = generate_masks_with_special_tokens_and_transfer_map(
        tokenized, model.specical_tokens, model.tokenizer)
 
    max_len = getattr(model, "max_text_len", 256)
    if self_masks.shape[1] > max_len:
        self_masks = self_masks[:, :max_len, :max_len]
        position_ids = position_ids[:, :max_len]
        for key in ("input_ids", "attention_mask", "token_type_ids"):
            if key in tokenized:
                tokenized[key] = tokenized[key][:, :max_len]
 
    tokenized_for_encoder = {k: v for k, v in tokenized.items() if k != "attention_mask"}
    tokenized_for_encoder["attention_mask"] = self_masks
    tokenized_for_encoder["position_ids"] = position_ids
    with torch.inference_mode():
        bert_output = model.bert(**tokenized_for_encoder)
        encoded = model.feat_map(bert_output["last_hidden_state"])
    valid = tokenized.attention_mask.bool()
    return {"encoded_text": encoded, "text_token_mask": valid,
            "position_ids": position_ids, "text_self_attention_masks": self_masks,
            "input_ids": tokenized.input_ids}
 
if MODEL is not None:
    TEXT_DICT = encode_text(MODEL, ["car ."])
else:
    TEXT_DICT = {"encoded_text": torch.randn(1, 4, 256),
                 "text_token_mask": torch.tensor([[True, True, True, True]])}
print("encoded text:", tuple(TEXT_DICT["encoded_text"].shape),
      "valid tokens:", int(TEXT_DICT["text_token_mask"].sum()))


In [ ]:
MAX_LEN = getattr(MODEL, "max_text_len", 256)
HIDDEN = MODEL.hidden_dim
 
assert tuple(TEXT_DICT["encoded_text"].shape) == (1, 4, HIDDEN)
 
long_out = encode_text(MODEL, [" ".join(["car"] * 300) + " ."])
assert tuple(long_out["encoded_text"].shape) == (1, MAX_LEN, HIDDEN), long_out["encoded_text"].shape
assert tuple(long_out["text_self_attention_masks"].shape) == (1, MAX_LEN, MAX_LEN)
assert tuple(long_out["position_ids"].shape) == (1, MAX_LEN)
assert tuple(long_out["text_token_mask"].shape) == (1, MAX_LEN)
 
for bad in ([], [""], ["   "]):
    try:
        encode_text(MODEL, bad)
        raise AssertionError(f"empty input {bad!r} was accepted")
    except ValueError:
        pass
 
print(f"Task 2 tests passed: normal (1,4,{HIDDEN}), truncated (1,{MAX_LEN},{HIDDEN}), empty input rejected")


In [ ]:
import csv
 
EVID_CAPTIONS = ["car .", "yellow car .", "red car . white car ."]
EVID = encode_text(MODEL, EVID_CAPTIONS)
ids = EVID["input_ids"].cpu()
valid = EVID["text_token_mask"].cpu()
pos = EVID["position_ids"].cpu()
masks = EVID["text_self_attention_masks"].cpu()
L = ids.shape[1]
print(f"batch of {len(EVID_CAPTIONS)} captions padded to length {L};",
      "encoded_text:", tuple(EVID["encoded_text"].shape))
 
TOKEN_TABLE = []
for b, cap in enumerate(EVID_CAPTIONS):
    toks = MODEL.tokenizer.convert_ids_to_tokens(ids[b].tolist())
    n_real = len(MODEL.tokenizer(cap)["input_ids"])

    assert int(valid[b].sum()) == n_real
    assert all(t == "[PAD]" for t in toks[n_real:])
    assert not valid[b, n_real:].any()
    print(f"\n{cap!r}  real tokens={n_real}  padded={L - n_real}")
    print(f"  {'col':>3} {'token':>8} {'id':>6} {'valid':>6} {'pos':>4}")
    for j, (t, i, v, p) in enumerate(zip(toks, ids[b].tolist(), valid[b].tolist(), pos[b].tolist())):
        print(f"  {j:>3} {t:>8} {i:>6} {str(v):>6} {p:>4}")
        TOKEN_TABLE.append({"caption": cap, "column": j, "token": t, "id": i,
                            "valid": v, "position_id": p})
 
with open(OUTPUT / "text_token_table.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(TOKEN_TABLE[0]))
    writer.writeheader(); writer.writerows(TOKEN_TABLE)
 
fig, axes = plt.subplots(1, len(EVID_CAPTIONS) + 1, figsize=(4.2 * (len(EVID_CAPTIONS) + 1), 4))
for ax, b, cap in zip(axes, range(len(EVID_CAPTIONS)), EVID_CAPTIONS):
    toks = MODEL.tokenizer.convert_ids_to_tokens(ids[b].tolist())
    ax.imshow(masks[b].float(), cmap="gray", vmin=0, vmax=1)
    ax.set_xticks(range(L)); ax.set_xticklabels(toks, rotation=90, fontsize=8)
    ax.set_yticks(range(L)); ax.set_yticklabels(toks, fontsize=8)
    ax.set_title(f"self-attention: {cap!r}", fontsize=9)
ax = axes[-1]
ax.imshow(valid.float(), cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
ax.set_yticks(range(len(EVID_CAPTIONS))); ax.set_yticklabels(EVID_CAPTIONS, fontsize=8)
ax.set_xticks(range(L)); ax.set_xlabel("token column")
ax.set_title("text_token_mask (green=real, red=[PAD])", fontsize=9)
fig.suptitle("Text encoder: special-token attention masks and padding validity")
fig.tight_layout()
fig.savefig(OUTPUT / "text_masks.png", dpi=170, bbox_inches="tight")
plt.show()
print("saved:", OUTPUT / "text_masks.png", "and", OUTPUT / "text_token_table.csv")

### Task 3 — ROIAlign visual exemplar tokens (15%)

Use pixel-space exemplar boxes and `roi_align` with `output_size=(1,1)`, `spatial_scale=1/8`, and `aligned=True`, then reshape the result to `(B,N,D)`.


In [ ]:
def extract_exemplar_tokens(combined_features, exemplar_boxes):
    if combined_features.ndim != 4 or len(exemplar_boxes) != combined_features.shape[0]:
        raise ValueError("Expected features (B,D,H,W) and one (N,4) box tensor per image")
    counts = [len(boxes) for boxes in exemplar_boxes]
    if len(set(counts)) != 1 or counts[0] == 0:
        raise ValueError("This exercise expects the same positive exemplar count per image")
    tokens = roi_align(combined_features, boxes=exemplar_boxes, output_size=(1, 1),
                       spatial_scale=1/8, aligned=True).squeeze(-1).squeeze(-1)
    return tokens.reshape(combined_features.shape[0], counts[0], -1)

EXEMPLAR_TOKENS = extract_exemplar_tokens(COMBINED_FEATURES, [car_boxes_resized.to(COMBINED_FEATURES.device)])
print("resized exemplar:", car_boxes_resized.tolist())
print("ROIAlign exemplar token:", tuple(EXEMPLAR_TOKENS.shape))


### Task 4 — Similarity matrix calculation (15%)

Compute query–prompt dot products, mask padded prompt columns with `-inf`, and visualize the valid part of the similarity matrix.


In [ ]:
def calculate_similarity_matrix(query_features, prompt_features, prompt_mask):
    if query_features.ndim != 3 or prompt_features.ndim != 3:
        raise ValueError("Expected query (B,N,D) and prompt (B,T,D)")
    if query_features.shape[0] != prompt_features.shape[0] or query_features.shape[-1] != prompt_features.shape[-1]:
        raise ValueError("Batch and feature dimensions must agree")
    if prompt_mask.shape != prompt_features.shape[:2]:
        raise ValueError("Prompt mask must have shape (B,T)")
    logits = query_features @ prompt_features.transpose(-1, -2)
    return logits.masked_fill(~prompt_mask[:, None, :], float("-inf"))

IMAGE_TOKENS = COMBINED_FEATURES.flatten(2).transpose(1, 2)
PROMPT_TOKENS = torch.cat([TEXT_DICT["encoded_text"].to(IMAGE_TOKENS.device), EXEMPLAR_TOKENS], dim=1)
PROMPT_MASK = torch.cat([TEXT_DICT["text_token_mask"].to(IMAGE_TOKENS.device),
                         torch.ones((1, EXEMPLAR_TOKENS.shape[1]), dtype=torch.bool, device=IMAGE_TOKENS.device)], dim=1)
SIMILARITY = calculate_similarity_matrix(IMAGE_TOKENS, PROMPT_TOKENS, PROMPT_MASK)

text_tokens = MODEL.tokenizer.convert_ids_to_tokens(TEXT_DICT["input_ids"][0].tolist())
assert text_tokens == ["[CLS]", "car", ".", "[SEP]"]
assert EXEMPLAR_TOKENS.shape[1] == 1

SIMILARITY_NO_CLS = SIMILARITY.clone()
SIMILARITY_NO_CLS[:, :, 0] = float("-inf")

SIMILARITY_NO_CLS_SEP = SIMILARITY.clone()
SIMILARITY_NO_CLS_SEP[:, :, [0, 3]] = float("-inf")

SIMILARITY_CAR_EXEMPLAR = SIMILARITY.clone()
SIMILARITY_CAR_EXEMPLAR[:, :, [0, 2, 3]] = float("-inf")

print("similarity matrix:", tuple(SIMILARITY.shape))
plt.figure(figsize=(7, 3)); plt.imshow(SIMILARITY[0, :200].detach().cpu(), aspect="auto", cmap="viridis")
plt.xlabel("prompt-token column"); plt.ylabel("image-token row (first 200)"); plt.colorbar(); plt.show()

labels = MODEL.tokenizer.convert_ids_to_tokens(TEXT_DICT["input_ids"][0].tolist()) + ["exemplar"]
h, w = COMBINED_FEATURES.shape[-2:]
heatmaps = SIMILARITY[0].detach().cpu().reshape(h, w, 5)

fig, axes = plt.subplots(1, 5, figsize=(16, 3), constrained_layout=True)
for i in range(5):
    im = axes[i].imshow(heatmaps[:, :, i], cmap="viridis",
                        vmin=heatmaps[torch.isfinite(heatmaps)].min().item(), vmax=heatmaps.max().item())
    axes[i].set_title(labels[i])
    axes[i].axis("off")
fig.colorbar(im, ax=axes.tolist(), label="Dot-product score", shrink=0.8)
fig.savefig(OUTPUT / "task4_prompt_token_heatmaps.png", dpi=150)
plt.show()

### Task 5 — Top-K encoder proposal selection (15%)

Reduce each proposal to its strongest prompt score, apply `topk` over proposals, and report the winning proposal IDs and prompt IDs.


In [ ]:
def select_topk(similarity, k):
    if similarity.ndim != 3 or not 1 <= k <= similarity.shape[1]:
        raise ValueError("Expected (B,N,T) similarities and 1 <= k <= N")
    proposal_scores, prompt_ids = similarity.max(dim=-1)
    scores, proposal_ids = torch.topk(proposal_scores, k, dim=1)
    selected_prompt_ids = torch.gather(prompt_ids, 1, proposal_ids)
    return scores, proposal_ids, selected_prompt_ids

TOPK_SCORES, TOPK_PROPOSALS, TOPK_PROMPTS = select_topk(SIMILARITY, 10)
token_names = text_tokens + ["exemplar"]

assert TOPK_SCORES.shape == TOPK_PROPOSALS.shape == TOPK_PROMPTS.shape == (1, 10)
assert torch.all(TOPK_SCORES[:, :-1] >= TOPK_SCORES[:, 1:])
expected_prompt_ids = SIMILARITY.argmax(dim=-1).gather(1, TOPK_PROPOSALS)
assert torch.equal(TOPK_PROMPTS, expected_prompt_ids)
selected_rows = SIMILARITY.gather(
    1, TOPK_PROPOSALS[:, :, None].expand(-1, -1, SIMILARITY.shape[-1])
)
expected_scores = selected_rows.gather(2, TOPK_PROMPTS[:, :, None]).squeeze(-1)
assert torch.allclose(TOPK_SCORES, expected_scores)
assert int(TOPK_PROMPTS.max()) < len(token_names)

print("Task 5 — canonical Top-K proposal ranking (K=10)")
print(f"{'rank':>4} {'score':>12} {'proposal_id':>12} {'prompt_id':>10}  prompt")
for rank, (score, proposal_id, prompt_id) in enumerate(
    zip(TOPK_SCORES[0], TOPK_PROPOSALS[0], TOPK_PROMPTS[0]), start=1
):
    prompt_id = int(prompt_id)
    print(
        f"{rank:>4} {float(score):>12.2f} {int(proposal_id):>12} "
        f"{prompt_id:>10}  {token_names[prompt_id]}"
    )

experiments = {
    "All tokens": SIMILARITY,
    "No CLS": SIMILARITY_NO_CLS,
    "No CLS/SEP": SIMILARITY_NO_CLS_SEP,
    "Car + exemplar": SIMILARITY_CAR_EXEMPLAR,
}
results = {}

print(f"\n{'Setting':<18} {'K':>3} {'Min score':>12} {'Overlap':>10}  Winning tokens")
for k in [5, 10, 20, 50]:
    _, original_ids, _ = select_topk(SIMILARITY, k)
    original_ids = set(original_ids[0].tolist())

    for name, similarity in experiments.items():
        scores, proposal_ids, prompt_ids = select_topk(similarity, k)
        assert torch.all(scores[:, :-1] >= scores[:, 1:])

        results[(name, k)] = {
            "scores": scores,
            "proposal_ids": proposal_ids,
            "prompt_ids": prompt_ids,
        }
        overlap = len(original_ids & set(proposal_ids[0].tolist()))
        counts = torch.bincount(prompt_ids[0], minlength=len(token_names))
        winners = ", ".join(
            f"{token_names[i]}: {count}"
            for i, count in enumerate(counts.tolist())
            if count > 0
        )
        print(
            f"{name:<18} {k:>3} {scores[0, -1].item():>12.2f} "
            f"{str(overlap) + '/' + str(k):>10}  {winners}"
        )
    print()

print(
    "Interpretation: each proposal is scored by its strongest prompt column; "
    "the reported prompt ID identifies the token that supplied that maximum."
)

### Task 6 — Shared integration experiment and results (25%)

Run the official model on all four prompt cases and include counts, localization plots at threshold `0.23`, component shapes, one ablation, runtime/hardware, and one failure case.

**TODO 6A — Prompt sensitivity experiment**

- create new text-only cases without changing the source images: `women → man` and `car → yellow car`;
- run each changed prompt through the live CountGD model using the same preprocessing and post-processing;
- report the changed captions, counts, scores, boxes, and localization plots; and
- compare each changed-prompt result with its original-prompt baseline.

Changed prompts require a live model forward pass. Run fresh inference for `man` and `yellow car`; do not reuse predictions from another query.


In [ ]:
import platform

def run_countgd(model, image_name, query="", exemplar_boxes=()):
    case_name = f"{Path(image_name).stem}_{query or 'exemplar'}".replace(" ", "_")
    case = prepare_case(case_name, image_name, text=query, boxes=exemplar_boxes)
    image_tensor, boxes = preprocess_case(case)
    with torch.inference_mode():
        output = model(
            image_tensor.unsqueeze(0).to(DEVICE),
            [boxes.to(DEVICE)],
            [torch.tensor([0], device=DEVICE)],
            captions=[case["caption"]],
        )
    assert tuple(output["pred_logits"].shape[1:]) == (900, 256)
    assert tuple(output["pred_boxes"].shape[1:]) == (900, 4)
    raw = {
        "confidences": output["pred_logits"][0].sigmoid().cpu().numpy().astype(np.float32),
        "boxes": output["pred_boxes"][0].cpu().numpy().astype(np.float32),
        "image_size": (case["image"].height, case["image"].width),
    }
    return case, raw

def postprocess_countgd(raw, threshold=0.23):
    confidences, boxes = raw["confidences"], raw["boxes"]
    finite = np.isfinite(confidences).all(1) & np.isfinite(boxes).all(1)
    scores = np.where(finite[:, None], confidences, -np.inf).max(1)
    keep = finite & (scores > threshold)
    scores, boxes = scores[keep], boxes[keep]
    h, w = raw["image_size"]
    cx, cy, bw, bh = boxes.T
    xyxy = np.column_stack([
        (cx-bw/2)*w, (cy-bh/2)*h, (cx+bw/2)*w, (cy+bh/2)*h
    ])
    xyxy[:, [0, 2]] = np.clip(xyxy[:, [0, 2]], 0, w-1)
    xyxy[:, [1, 3]] = np.clip(xyxy[:, [1, 3]], 0, h-1)
    centers = np.column_stack([
        (xyxy[:, 0]+xyxy[:, 2])/2, (xyxy[:, 1]+xyxy[:, 3])/2
    ])
    return {"scores": scores, "boxes": xyxy, "centers": centers, "count": len(scores)}

def predict_countgd(image_name, query="", exemplar_boxes=(), threshold=0.23):
    case, raw = run_countgd(MODEL, image_name, query, exemplar_boxes)
    return {"case": case, "raw": raw, "result": postprocess_countgd(raw, threshold)}

def synchronize_device():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

OFFICIAL_THRESHOLD = 0.23
REQUESTS = [
    ("women_text", "women.jpg", "women", (), 5),
    ("car_text", "car.jpg", "car", (), 16),
    ("car_exemplar", "car.jpg", "", [[55, 72, 151, 218]], 16),
    ("car_multimodal", "car.jpg", "car", [[55, 72, 151, 218]], 16),
]

processor = platform.processor() or platform.machine() or "unknown CPU"
if DEVICE.type == "cuda":
    device_name = torch.cuda.get_device_name(DEVICE)
else:
    device_name = processor
print("Task 6 runtime environment")
print("Python:", platform.python_version(), "| PyTorch:", torch.__version__)
print("device:", DEVICE, "| name:", device_name)
print("CUDA available:", torch.cuda.is_available(), "| PyTorch CUDA:", torch.version.cuda)
print("logical CPUs:", os.cpu_count(), "| platform:", platform.platform())
print("checkpoint SHA-256:", CHECKPOINT_SHA256)

PREDICTIONS = {}
for name, image_name, query, exemplar_boxes, expected_count in REQUESTS:
    synchronize_device()
    t0 = time.perf_counter()
    prediction = predict_countgd(
        image_name, query, exemplar_boxes, threshold=OFFICIAL_THRESHOLD
    )
    synchronize_device()
    prediction["runtime_seconds"] = time.perf_counter() - t0
    prediction["expected_count"] = expected_count
    PREDICTIONS[name] = prediction

expected = {name: expected_count for name, _, _, _, expected_count in REQUESTS}
actual = {name: item["result"]["count"] for name, item in PREDICTIONS.items()}
assert actual == expected, (actual, expected)

RAW = {name: value["raw"] for name, value in PREDICTIONS.items()}
RESULTS = {name: value["result"] for name, value in PREDICTIONS.items()}

print("\nOfficial live baselines — max-token score > 0.23")
print(
    f"{'case':<18} {'mode':<17} {'caption':<12} {'raw logits':<12} "
    f"{'raw boxes':<10} {'expected':>8} {'count':>6} {'mean':>8} {'time(s)':>8}"
)
for name, prediction in PREDICTIONS.items():
    case, raw, result = prediction["case"], prediction["raw"], prediction["result"]
    mean_score = float(result["scores"].mean()) if result["count"] else float("nan")
    mode = case["mode"]
    if name == "car_exemplar":
        mode = "exemplar-guided"
    print(
        f"{name:<18} {mode:<17} {case['caption']!r:<12} "
        f"{str(raw['confidences'].shape):<12} {str(raw['boxes'].shape):<10} "
        f"{prediction['expected_count']:>8} {result['count']:>6} "
        f"{mean_score:>8.3f} {prediction['runtime_seconds']:>8.2f}"
    )

print("\nIntegration component shapes")
print("image batch:", tuple(car_batch.shape))
print("backbone levels:", [tuple(feature.decompose()[0].shape) for feature in IMAGE_FEATURES])
print("combined image map:", tuple(COMBINED_FEATURES.shape))
print("encoded text:", tuple(TEXT_DICT["encoded_text"].shape))
print("exemplar tokens:", tuple(EXEMPLAR_TOKENS.shape))
print("similarity matrix:", tuple(SIMILARITY.shape))
print("Task 5 Top-K:", tuple(TOPK_SCORES.shape))
print("model outputs: confidences=(900, 256), boxes=(900, 4)")

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (name, prediction) in zip(axes.flat, PREDICTIONS.items()):
    case, result = prediction["case"], prediction["result"]
    ax.imshow(case["image"])
    for score, (x1, y1, x2, y2), (x, y) in zip(
        result["scores"], result["boxes"], result["centers"]
    ):
        ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, ec="lime", lw=1.2))
        ax.plot(x, y, "r.", ms=4)
    for x1, y1, x2, y2 in case["boxes"]:
        ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, ec="gold", lw=3))
    shown_mode = "exemplar-guided" if name == "car_exemplar" else case["mode"]
    ax.set_title(f"{name} | {shown_mode} | count={result['count']}")
    ax.axis("off")
fig.suptitle("Live CountGD detections — official max-token score > 0.23")
fig.tight_layout()
figure_path = OUTPUT / "official_countgd_car_women_tests.png"
fig.savefig(figure_path, dpi=170, bbox_inches="tight")
plt.show()
print("saved:", figure_path)

print("\nModality-ablation interpretation")
for name in ("car_text", "car_exemplar", "car_multimodal"):
    print(f"{name}: {RESULTS[name]['count']} detections at the same 0.23 threshold")
print(
    "All three car modes reproduce the expected count, but equal counts alone do not "
    "prove that their confidence scores or localization boxes are identical. The "
    "exemplar-guided case uses the structural caption 'object .' plus the visual token."
)

In [ ]:
PROMPT_VARIANTS = [
    {
        "name": "women_as_man",
        "image_name": "women.jpg",
        "original_key": "women_text",
        "original_prompt": "women",
        "changed_prompt": "man",
    },
    {
        "name": "car_as_yellow_car",
        "image_name": "car.jpg",
        "original_key": "car_text",
        "original_prompt": "car",
        "changed_prompt": "yellow car",
    },
]

def phrase_token_indices(model, caption):
    encoded = model.tokenizer(caption)
    tokens = model.tokenizer.convert_ids_to_tokens(encoded["input_ids"])
    ignored = set(model.tokenizer.all_special_tokens) | {".", ",", ";", ":", "!", "?"}
    indices = [index for index, token in enumerate(tokens) if token not in ignored]
    if not indices:
        raise ValueError(f"No meaningful text tokens found in {caption!r}")
    return indices, [tokens[index] for index in indices]

def postprocess_phrase_query(raw, model, caption, threshold=0.50):
    indices, tokens = phrase_token_indices(model, caption)
    phrase_scores = raw["confidences"][:, indices].prod(axis=1, dtype=np.float32)
    phrase_raw = {**raw, "confidences": phrase_scores[:, None]}
    result = postprocess_countgd(phrase_raw, threshold)
    result["phrase_tokens"] = tokens
    return result

def evaluate_prompt_variants(model, prompt_specs, threshold=OFFICIAL_THRESHOLD):
    if model is None:
        raise RuntimeError("Changed prompts require fresh live-model inference.")
    evaluated = {}
    for spec in prompt_specs:
        synchronize_device()
        t0 = time.perf_counter()
        case, raw = run_countgd(model, spec["image_name"], spec["changed_prompt"])
        synchronize_device()
        seconds = time.perf_counter() - t0
        official_result = postprocess_countgd(raw, threshold)
        phrase_result = postprocess_phrase_query(
            raw, model, case["caption"], threshold=0.50
        )
        evaluated[spec["name"]] = {
            "case": case,
            "raw": raw,
            "result": official_result,
            "phrase_result": phrase_result,
            "runtime_seconds": seconds,
            "spec": spec,
        }
    return evaluated

PROMPT_RESULTS = evaluate_prompt_variants(MODEL, PROMPT_VARIANTS)
print("Task 6A — fresh changed-prompt inference using official max-token score > 0.23")
for spec in PROMPT_VARIANTS:
    record = PROMPT_RESULTS[spec["name"]]
    phrase_tokens = record["phrase_result"]["phrase_tokens"]
    print(
        spec["name"], repr(record["case"]["caption"]),
        "tokens=", phrase_tokens,
        "official_count=", record["result"]["count"],
        f"runtime={record['runtime_seconds']:.2f}s",
    )

In [ ]:
import json

def draw_detection_result(ax, case, result, color="cyan", annotate_scores=True):
    ax.imshow(case["image"])
    for score, (x1, y1, x2, y2), (x, y) in zip(
        result["scores"], result["boxes"], result["centers"]
    ):
        ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False, ec=color, lw=1.5))
        if annotate_scores:
            ax.text(
                x, y, f"{score:.2f}", color="black", fontsize=6,
                ha="center", va="center",
                bbox=dict(facecolor=color, alpha=0.75, pad=0.7, lw=0),
            )
    ax.axis("off")

COMPARISON_SUMMARY = []
DETECTION_ROWS = []
PHRASE_ABLATION_ROWS = []
fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)

for row, spec in enumerate(PROMPT_VARIANTS):
    original = PREDICTIONS[spec["original_key"]]
    changed = PROMPT_RESULTS[spec["name"]]
    pair_records = [
        ("original", spec["original_prompt"], original),
        ("changed", spec["changed_prompt"], changed),
    ]

    original_count = original["result"]["count"]
    changed_count = changed["result"]["count"]
    for col, (role, prompt, record) in enumerate(pair_records):
        result, case = record["result"], record["case"]
        h, w = record["raw"]["image_size"]
        assert result["count"] == len(result["scores"]) == len(result["boxes"])
        if result["count"]:
            assert np.all((result["boxes"][:, [0, 2]] >= 0) & (result["boxes"][:, [0, 2]] <= w-1))
            assert np.all((result["boxes"][:, [1, 3]] >= 0) & (result["boxes"][:, [1, 3]] <= h-1))

        COMPARISON_SUMMARY.append({
            "image": spec["image_name"],
            "role": role,
            "prompt": prompt,
            "scoring_rule": "max token confidence",
            "threshold": OFFICIAL_THRESHOLD,
            "count": int(result["count"]),
            "count_delta_from_original": int(result["count"] - original_count),
            "runtime_seconds": round(float(record["runtime_seconds"]), 3),
            "live_inference": True,
        })

        order = np.argsort(-result["scores"])
        for rank, index in enumerate(order, start=1):
            x1, y1, x2, y2 = result["boxes"][index]
            DETECTION_ROWS.append({
                "image": spec["image_name"],
                "role": role,
                "prompt": prompt,
                "rank": rank,
                "score": round(float(result["scores"][index]), 6),
                "x1": round(float(x1), 2),
                "y1": round(float(y1), 2),
                "x2": round(float(x2), 2),
                "y2": round(float(y2), 2),
                "threshold": OFFICIAL_THRESHOLD,
                "live_inference": True,
            })

        draw_detection_result(axes[row, col], case, result)
        axes[row, col].set_title(
            f"{role}: {prompt!r} | count={result['count']} | max-token > 0.23",
            fontsize=10,
        )

    original_phrase = postprocess_phrase_query(
        original["raw"], MODEL, f"{spec['original_prompt']} .", threshold=0.50
    )
    changed_phrase = changed["phrase_result"]
    for role, prompt, phrase_result in (
        ("original", spec["original_prompt"], original_phrase),
        ("changed", spec["changed_prompt"], changed_phrase),
    ):
        PHRASE_ABLATION_ROWS.append({
            "image": spec["image_name"],
            "role": role,
            "prompt": prompt,
            "tokens": " ".join(phrase_result["phrase_tokens"]),
            "scoring_rule": "product of meaningful-token confidences",
            "threshold": 0.50,
            "count": int(phrase_result["count"]),
        })

fig.suptitle(
    "Fresh live CountGD prompt sensitivity — official max-token score > 0.23",
    fontsize=13,
)
comparison_figure = OUTPUT / "countgd_prompt_comparison.png"
fig.savefig(comparison_figure, dpi=170, bbox_inches="tight")
plt.show()

print("Official original-versus-changed comparison")
print(f"{'image':<10} {'role':<9} {'prompt':<12} {'count':>6} {'delta':>7} {'time(s)':>8}")
for item in COMPARISON_SUMMARY:
    print(
        f"{item['image']:<10} {item['role']:<9} {item['prompt']:<12} "
        f"{item['count']:>6} {item['count_delta_from_original']:>7} "
        f"{item['runtime_seconds']:>8.3f}"
    )

print("\nChanged-prompt detections — official scores and pixel xyxy boxes")
print(f"{'prompt':<12} {'rank':>4} {'score':>8} {'x1':>8} {'y1':>8} {'x2':>8} {'y2':>8}")
for item in DETECTION_ROWS:
    if item["role"] == "changed":
        print(
            f"{item['prompt']:<12} {item['rank']:>4} {item['score']:>8.3f} "
            f"{item['x1']:>8.1f} {item['y1']:>8.1f} {item['x2']:>8.1f} {item['y2']:>8.1f}"
        )

print("\nSeparate phrase-product ablation — meaningful-token product > 0.50")
print(f"{'image':<10} {'role':<9} {'prompt':<12} {'tokens':<16} {'count':>6}")
for item in PHRASE_ABLATION_ROWS:
    print(
        f"{item['image']:<10} {item['role']:<9} {item['prompt']:<12} "
        f"{item['tokens']:<16} {item['count']:>6}"
    )

comparison_json = OUTPUT / "table3_prompt_comparison.json"
comparison_csv = OUTPUT / "table3_prompt_detections.csv"
with open(comparison_json, "w") as stream:
    json.dump({
        "official_summary": COMPARISON_SUMMARY,
        "official_detections": DETECTION_ROWS,
        "phrase_product_ablation": PHRASE_ABLATION_ROWS,
    }, stream, indent=2)
with open(comparison_csv, "w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(DETECTION_ROWS[0]))
    writer.writeheader()
    writer.writerows(DETECTION_ROWS)

print("\nsaved:", comparison_figure)
print("saved:", comparison_json)
print("saved:", comparison_csv)
print(
    "Interpretation: the official comparison holds preprocessing and max-token "
    "post-processing fixed at 0.23. The phrase-product rows are a separate ablation, "
    "so their counts must not be reported as official CountGD baseline counts."
)

In [ ]:
YELLOW_CAR_VISIBLE_REFERENCE = 2
yellow_record = PROMPT_RESULTS["car_as_yellow_car"]
yellow_official = yellow_record["result"]
yellow_phrase = yellow_record["phrase_result"]
failure_observed = yellow_official["count"] != YELLOW_CAR_VISIBLE_REFERENCE

print("Task 6 failure analysis — yellow-car modifier sensitivity")
print("manual visible reference (yellow cars):", YELLOW_CAR_VISIBLE_REFERENCE)
print("official max-token > 0.23 count:", yellow_official["count"])
print("phrase-product > 0.50 diagnostic count:", yellow_phrase["count"])
print("observed official-count failure:", failure_observed)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)
draw_detection_result(axes[0], yellow_record["case"], yellow_official, color="orange", annotate_scores=False)
axes[0].set_title(
    f"Official max-token > 0.23\ncount={yellow_official['count']} "
    f"(visible yellow reference={YELLOW_CAR_VISIBLE_REFERENCE})"
)
draw_detection_result(axes[1], yellow_record["case"], yellow_phrase, color="lime", annotate_scores=True)
axes[1].set_title(
    f"Diagnostic token-product > 0.50\ncount={yellow_phrase['count']}"
)
fig.suptitle("Failure case: max-token aggregation can ignore the colour modifier")
failure_figure = OUTPUT / "failure_yellow_car_modifier.png"
fig.savefig(failure_figure, dpi=170, bbox_inches="tight")
plt.show()
print("saved:", failure_figure)

if failure_observed:
    print(
        "Interpretation: the official max-token rule counts proposals when any token "
        "is confident, so confidence for 'car' can dominate even when 'yellow' is weak. "
        "This run therefore counts non-yellow cars for the compound prompt. The phrase "
        "product is shown only as a diagnostic; this single image does not establish "
        "that it is a generally better post-processing rule."
    )
else:
    print(
        "Interpretation: this live run did not reproduce a count error for the selected "
        "failure probe. Box-level differences should be inspected before making a failure claim."
    )

print("\nCar modality ablation at the shared official threshold 0.23")
print(f"{'mode':<18} {'count':>6} {'min score':>10} {'mean score':>11} {'max score':>10}")
for name in ("car_text", "car_exemplar", "car_multimodal"):
    scores = RESULTS[name]["scores"]
    print(
        f"{name:<18} {RESULTS[name]['count']:>6} {float(scores.min()):>10.3f} "
        f"{float(scores.mean()):>11.3f} {float(scores.max()):>10.3f}"
    )
print(
    "Interpretation: the three prompt modes have equal counts on this image, while their "
    "score distributions provide the quantitative ablation evidence beyond count alone."
)

In [ ]:
from PIL import Image

HUE_SHIFT_DEG = -50          # yellow (~50 deg) -> red (~0 deg). Greys/whites barely change.

def product_scores(raw, query):
    toks = MODEL.tokenizer.convert_ids_to_tokens(MODEL.tokenizer(f"{query} .")["input_ids"])
    keep = [i for i, t in enumerate(toks)
            if t not in MODEL.tokenizer.all_special_tokens and t not in {".", ","}]
    conf = raw["confidences"]
    return conf.max(1), conf[:, keep].prod(1)

def to_xyxy(raw, idx):
    h, w = raw["image_size"]
    cx, cy, bw, bh = raw["boxes"][idx].T
    return np.column_stack([(cx - bw/2) * w, (cy - bh/2) * h, (cx + bw/2) * w, (cy + bh/2) * h])

def iou(a, b):
    x1, y1 = np.maximum(a[0], b[0]), np.maximum(a[1], b[1])
    x2, y2 = np.minimum(a[2], b[2]), np.minimum(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter + 1e-9)

original = Image.open(ASSETS / "car.jpg").convert("RGB")
h, s, v = original.convert("HSV").split()
shift = int(round(HUE_SHIFT_DEG / 360 * 256)) % 256
shifted = Image.merge("HSV", (h.point(lambda x: (x + shift) % 256), s, v)).convert("RGB")
SHIFTED_PATH = (OUTPUT / "car_hue_shifted.jpg").resolve()
shifted.save(SHIFTED_PATH, quality=95)

_, raw_ref = run_countgd(MODEL, "car.jpg", "yellow car")
_, p_ref = product_scores(raw_ref, "yellow car")
REF_BOXES = to_xyxy(raw_ref, np.argsort(-p_ref)[:2])

RUNS = [("original", "car.jpg",          "yellow car"),
        ("original", "car.jpg",          "red car"),
        ("shifted",  str(SHIFTED_PATH),  "red car"),
        ("shifted",  str(SHIFTED_PATH),  "yellow car")]

G_ROWS, G_TOP2 = [], []
for label, image_name, query in RUNS:
    _, raw = run_countgd(MODEL, image_name, query)
    s_max, s_prod = product_scores(raw, query)
    order = np.argsort(-s_prod)
    top2 = to_xyxy(raw, order[:2])
    hits = sum(max(iou(b, r) for r in REF_BOXES) > 0.5 for b in top2)  
    G_TOP2.append((label, image_name, query, top2))
    G_ROWS.append({"image": label, "prompt": query,
                   "max@0.23": int((s_max > 0.23).sum()), "prod@0.50": int((s_prod > 0.50).sum()),
                   "top-2 are target cars": f"{hits}/2",
                   "gap #2-#3 (product)": round(float(s_prod[order[1]] - s_prod[order[2]]), 3)})

print(f"Hue-shift test (shift {HUE_SHIFT_DEG} deg). Target = the two cars that are yellow in the original.")
print(f"{'image':<9} {'prompt':<11} {'max@0.23':>8} {'prod@0.50':>9} {'top-2 target':>12} {'gap #2-#3':>9}")
for r in G_ROWS:
    print(f"{r['image']:<9} {r['prompt']:<11} {r['max@0.23']:>8} {r['prod@0.50']:>9} "
          f"{r['top-2 are target cars']:>12} {r['gap #2-#3 (product)']:>9}")

fig, axes = plt.subplots(1, len(G_TOP2), figsize=(5 * len(G_TOP2), 4), constrained_layout=True)
for ax, (label, image_name, query, top2) in zip(axes, G_TOP2):
    ax.imshow(original if label == "original" else shifted)
    for x1, y1, x2, y2 in top2:
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec="red", lw=3))
    ax.set_title(f'{label}: "{query}" (top-2)', fontsize=10)
    ax.axis("off")
fig.suptitle("Hue-shift test: does the model follow the word or the most distinct colour?")
fig.savefig(OUTPUT / "hue_shift_test.png", dpi=170, bbox_inches="tight")
plt.show()